# Norm Provenance Ledger v0.4.3 – расширенная проверка надёжности: 30 трасс, два судьи

Запускает обоих судей (GPT-5.6 Terra и GPT-5.6 Sol) на одних и тех же 30 трассах, каждого в своей коллекции. Ячейки 1–5 ничего не отправляют.

## 1. Установка SDK (версия закреплена)

In [ ]:
!pip -q install docent==0.1.87
import docent, importlib.metadata as im
print('docent', im.version('docent'))

## 2. Загрузка архива
После запуска нажми «Выбрать файлы» и выбери `norm_ledger_v043.zip`.

In [ ]:
from google.colab import files
import zipfile, os, pathlib
up = files.upload()
name = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(name).extractall('/content')
ROOT = pathlib.Path('/content/norm_ledger_v043')
os.chdir(ROOT)
print('рабочая папка:', ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

## 3. Офлайн-проверка (без сети)
Должно вывести 30 прогонов и 1652 блока.

In [ ]:
!python protocol/docent_run.py dry-run --corpus rel30

## 4. Ключ API
Способ 1 (надёжнее): иконка ключа слева → Secrets → добавить `DOCENT_API_KEY`, включить доступ для блокнота.
Способ 2: ячейка сама спросит ключ скрытым полем. Ключ не печатается и не сохраняется в блокноте.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['DOCENT_API_KEY'] = userdata.get('DOCENT_API_KEY')
    print('ключ взят из Secrets')
except Exception:
    from getpass import getpass
    os.environ['DOCENT_API_KEY'] = getpass('DOCENT_API_KEY: ')
    print('ключ введён вручную')
print('ключ задан:', bool(os.environ.get('DOCENT_API_KEY')))

## 5. Какие модели доступны
Первый список – судьи, доступные без собственного ключа провайдера, как они записаны в SDK. Второй – то, что сервер отдаёт для «readings» (может не совпадать со списком судей). Выбери одну модель и впиши её в ячейку 6 в виде `provider:model:effort`.

In [ ]:
from docent.sdk import Docent
from docent._llm_util.providers.preference_types import PUBLIC_PROVIDER_PREFERENCES
print('Судьи по умолчанию в SDK 0.1.87:')
for m in PUBLIC_PROVIDER_PREFERENCES.default_judge_models:
    print(f'  {m.provider}:{m.model_name}:{m.reasoning_effort}')
try:
    c = Docent(api_key=os.environ['DOCENT_API_KEY'])
    print('\nМодели на сервере (readings):')
    for m in c.get_reading_models():
        print('  ', {k: m.get(k) for k in ('provider','model_name','reasoning_effort','uses_byok')})
except Exception as e:
    print('список с сервера не получен:', type(e).__name__, str(e)[:200])

## 6. Запуск двух судей (сетевые записи)
Две коллекции: «v0.4.3-rel30-j1» и «v0.4.3-rel30-j2». Скачаются два файла состояния, сохрани оба.

In [ ]:
!python protocol/docent_run.py launch --corpus rel30 --judge "openai:gpt-5.6-terra:medium" --label v0.4.3-rel30-j1
!python protocol/docent_run.py launch --corpus rel30 --judge "openai:gpt-5.6-sol:medium" --label v0.4.3-rel30-j2
from google.colab import files
files.download('docent_pilot_state_v0.4.3-rel30-j1.json'); files.download('docent_pilot_state_v0.4.3-rel30-j2.json')

## 7. Ожидание результатов
Оба судьи параллельно, до 30 результатов у каждого. Длинные трассы ChatDev обрабатываются дольше.

In [ ]:
import json, time
c = Docent(api_key=os.environ['DOCENT_API_KEY'])
states = {j: json.load(open(f'docent_pilot_state_v0.4.3-rel30-{j}.json')) for j in ('j1','j2')}
for i in range(90):
    done = True
    for j, st in states.items():
        try:
            s = c.get_rubric_run_state(st['collection_id'], st['rubric_id'], include_failures=True)
            n = len(s.get('results') or [])
            print(time.strftime('%H:%M:%S'), j, 'результатов:', n, 'из 30')
            done = done and n >= 30
        except Exception as e:
            print(time.strftime('%H:%M:%S'), j, 'сервер не ответил:', type(e).__name__); done = False
    if done:
        print('готово'); break
    time.sleep(60)

## 8. Сырая выгрузка обоих судей
Пришли оба файла в чат.

In [ ]:
!python protocol/docent_run.py raw-export --corpus rel30 --label v0.4.3-rel30-j1
!python protocol/docent_run.py raw-export --corpus rel30 --label v0.4.3-rel30-j2
files.download('raw_run_state_v0.4.3-rel30-j1.json'); files.download('raw_run_state_v0.4.3-rel30-j2.json')

## Если Colab сбросился
Заново ячейки 1–4, затем загрузи оба сохранённых файла состояния в `/content/norm_ledger_v043/` (через `files.upload()`) и переходи к ячейкам 7–8. Ячейку 6 повторно не запускай.

## 9. Карта соответствия (без сетевых записей)
Docent в выгрузке называет прогоны своими внутренними ID. Эта ячейка спрашивает у Docent, какому исходному прогону соответствует каждый ID (поле source_run_id в метаданных), и сохраняет два маленьких файла. Пришли оба в чат.

In [ ]:
import json
c = Docent(api_key=os.environ['DOCENT_API_KEY'])
for j in ('j1', 'j2'):
    st = json.load(open(f'docent_pilot_state_v0.4.3-rel30-{j}.json'))
    raw = json.load(open(f'raw_run_state_v0.4.3-rel30-{j}.json'))
    ids = [r['agent_run_id'] for r in raw['results']]
    # плюс все прогоны коллекции, включая тот, у которого может не быть результата
    mp = {}
    for rid in ids:
        md = c.get_agent_run_metadata(st['collection_id'], rid)
        mp[rid] = md.get('source_run_id') or md.get('metadata', {}).get('source_run_id')
    json.dump(mp, open(f'id_map_v0.4.3-rel30-{j}.json', 'w'), indent=1)
    print(j, len(mp), 'сопоставлено; без source_run_id:', [k for k, v in mp.items() if not v])
    files.download(f'id_map_v0.4.3-rel30-{j}.json')